# Benchmark Veri Dogrulama
yfinance baglantisi, sembol verileri, gram altin/gumus donusumu ve normalizasyon testleri.
Her hucre bagimsiz calisir. PASS = OK, AssertionError veya hata = sorun var.

In [ ]:
# Hucre 1 - Kurulum ve lib import
import subprocess, sys, os

REQUIRED = ["yfinance", "plotly", "ipywidgets"]
for pkg in REQUIRED:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import yfinance as yf
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Jupyter'da __file__ yoktur; notebook her zaman repo kokunden acildigi icin
# os.getcwd() guvenli yol cozumu saglar.
LIB_PATH = os.path.join(os.getcwd(), "lib")
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from benchmark_engine import (
    normalize_to_100, build_benchmark_series, build_deposit_series,
    TROY_OZ_TO_GRAM, GRAM_SYMBOLS
)
from data_loader import fetch_prices, load_cpi_series, load_tcmb_rates
from chart_builder import build_performance_line_chart

print("Kurulum tamam")
print(f"TROY_OZ_TO_GRAM = {TROY_OZ_TO_GRAM}")
print(f"GRAM_SYMBOLS    = {GRAM_SYMBOLS}")

In [2]:
# Hucre 2 - Ham fiyat testi
# yfinance'ten son 5 gunluk veri cek, her sembol icin son fiyati yazdir
SEMBOLLER = ["GC=F", "SI=F", "USDTRY=X", "EURTRY=X", "XU100.IS"]

print("yfinance'ten veri cekiliyor...")
raw = yf.download(SEMBOLLER, period="5d", auto_adjust=True, progress=False)

if isinstance(raw.columns, pd.MultiIndex):
    raw = raw["Close"]

print()
print("=" * 45)
print(f"{'Sembol':<15} {'Son Fiyat':>12} {'Tarih':>12}")
print("=" * 45)

hata_var = False
for sym in SEMBOLLER:
    if sym in raw.columns:
        seri = raw[sym].dropna()
        if len(seri) > 0:
            son_fiyat = seri.iloc[-1]
            son_tarih = seri.index[-1].strftime("%Y-%m-%d")
            print(f"{sym:<15} {son_fiyat:>12.4f} {son_tarih:>12}")
        else:
            print(f"{sym:<15} {'VERİ YOK':>12} {'---':>12}  <-- KONTROL ET")
            hata_var = True
    else:
        print(f"{sym:<15} {'SUTUN YOK':>12} {'---':>12}  <-- KONTROL ET")
        hata_var = True

print("=" * 45)
if hata_var:
    print("UYARI: Bazi semboller veri getirmedi. Internet baglantisinizi kontrol edin.")
else:
    print("PASS: Tum semboller veri getirdi")

yfinance'ten veri cekiliyor...

Sembol             Son Fiyat        Tarih
GC=F               4707.0000   2026-05-10
SI=F                 80.7150   2026-05-10
USDTRY=X             45.3318   2026-05-11
EURTRY=X             53.3637   2026-05-11
XU100.IS          15062.7002   2026-05-08
PASS: Tum semboller veri getirdi


In [3]:
# Hucre 3 - Gram altin/gumus donusum testi
# GC=F: USD/troy oz  x  USDTRY  /  31.1035  =  TL/gram
gc_usd  = raw["GC=F"].dropna().iloc[-1]
si_usd  = raw["SI=F"].dropna().iloc[-1]
usdtry  = raw["USDTRY=X"].dropna().iloc[-1]

gram_altin_tl = gc_usd * usdtry / TROY_OZ_TO_GRAM
gram_gumus_tl = si_usd * usdtry / TROY_OZ_TO_GRAM

print("=" * 50)
print(f"GC=F (USD/troy oz)      : {gc_usd:.2f}")
print(f"SI=F (USD/troy oz)      : {si_usd:.4f}")
print(f"USDTRY                  : {usdtry:.4f}")
print(f"TROY_OZ_TO_GRAM         : {TROY_OZ_TO_GRAM}")
print("=" * 50)
print(f"Gram Altin (TL/gram)    : {gram_altin_tl:.2f} TL")
print(f"Gram Gumus (TL/gram)    : {gram_gumus_tl:.4f} TL")
print("=" * 50)

# Sanity check: mantikli araliklar mi?
assert 1500 < gram_altin_tl < 15000, (
    f"Gram altin fiyati beklenmedik: {gram_altin_tl:.2f} TL "
    f"(beklenen: 1500-15000 TL)"
)
assert 10 < gram_gumus_tl < 1000, (
    f"Gram gumus fiyati beklenmedik: {gram_gumus_tl:.4f} TL "
    f"(beklenen: 10-1000 TL)"
)
print("PASS: Gram donusum sanity check")

GC=F (USD/troy oz)      : 4707.00
SI=F (USD/troy oz)      : 80.7150
USDTRY                  : 45.3318
TROY_OZ_TO_GRAM         : 31.1035
Gram Altin (TL/gram)    : 6860.22 TL
Gram Gumus (TL/gram)    : 117.6381 TL
PASS: Gram donusum sanity check


In [4]:
# Hucre 4 - Normalizasyon assert testleri

# Test 1: temel normalizasyon
test_s = pd.Series(
    [1800.0, 2700.0, 3600.0],
    index=pd.date_range("2023-01-02", periods=3, freq="B")
)
result = normalize_to_100(test_s, "2023-01-02")
assert result.iloc[0] == 100.0, f"Ilk deger 100 olmali: {result.iloc[0]}"
assert abs(result.iloc[1] - 150.0) < 1e-9
assert abs(result.iloc[2] - 200.0) < 1e-9
print("PASS: normalize_to_100 temel test")

# Test 2: start_date indexte yok - nearest forward
test_s2 = pd.Series(
    [500.0, 1000.0],
    index=pd.to_datetime(["2023-01-04", "2023-01-05"])
)
result2 = normalize_to_100(test_s2, start_date="2023-01-02")  # hafta sonu
assert result2.iloc[0] == 100.0
assert abs(result2.iloc[1] - 200.0) < 1e-9
print("PASS: normalize_to_100 nearest-forward (hafta sonu basi)")

# Test 3: gram donusum benchmark_engine ile entegrasyon
idx = pd.date_range("2023-01-02", periods=3, freq="B")
test_prices = pd.DataFrame({"GC=F": [1800.0, 1900.0, 2000.0]}, index=idx)
test_fx     = pd.Series([27.0, 27.0, 27.0], index=idx)

bench = build_benchmark_series(
    symbols=["GC=F"],
    start_date="2023-01-02",
    end_date="2023-01-06",
    prices=test_prices,
    fx_usdtry=test_fx,
    currency="TL",
)
# Baslangic=100 olmali
assert abs(bench["GC=F"].iloc[0] - 100.0) < 1e-9
# Fiyat 1800 -> 2000: +11.11%, 100 -> 111.11
expected_last = 2000 / 1800 * 100
assert abs(bench["GC=F"].iloc[2] - expected_last) < 0.001
print(f"PASS: Gram altin benchmark normalizasyonu (son deger: {bench['GC=F'].iloc[2]:.2f}, beklenen: {expected_last:.2f})")

print()
print("Tum normalizasyon testleri GECTI")

PASS: normalize_to_100 temel test
PASS: normalize_to_100 nearest-forward (hafta sonu basi)
PASS: Gram altin benchmark normalizasyonu (son deger: 111.11, beklenen: 111.11)

Tum normalizasyon testleri GECTI


In [ ]:
# Hucre 5 - Son 30 gunluk canli benchmark grafigi
DRIVE_BASE = os.path.join(os.getcwd(), "data") + os.sep
CACHE_PATH = os.path.join(DRIVE_BASE, "cache")
os.makedirs(CACHE_PATH, exist_ok=True)

end_date   = datetime.today().strftime("%Y-%m-%d")
start_date = (datetime.today() - timedelta(days=45)).strftime("%Y-%m-%d")  # is gunu buffer

SYMBOLS = {
    "Gram Altin":  "GC=F",
    "Gram Gumus":  "SI=F",
    "DOLAR":       "USDTRY=X",
    "EURO":        "EURTRY=X",
    "BIST100":     "XU100.IS",
}

all_symbols = list(SYMBOLS.values()) + ["USDTRY=X"]
prices = fetch_prices(all_symbols, start=start_date, end=end_date, cache_path=CACHE_PATH)
fx_usdtry = prices["USDTRY=X"].dropna()

# Gercek 30 gun baslangicini hesapla
biz_days = prices["USDTRY=X"].dropna().index
actual_start = biz_days[-min(30, len(biz_days))].strftime("%Y-%m-%d")

benchmark_df = build_benchmark_series(
    symbols=list(SYMBOLS.values()),
    start_date=actual_start,
    end_date=end_date,
    prices=prices,
    fx_usdtry=fx_usdtry,
    currency="TL",
)

# Getiri tablosu
son = benchmark_df.dropna(how="all").iloc[-1].dropna()
sym_to_name = {v: k for k, v in SYMBOLS.items()}
print("\nSon 30 Gun Getiri Ozeti (TL, baz=100)")
print("=" * 38)
for sym, val in son.sort_values(ascending=False).items():
    isim = sym_to_name.get(sym, sym)
    isaretli = f"+{val-100:.2f}%" if val >= 100 else f"{val-100:.2f}%"
    print(f"  {isim:<14}: {val:6.2f}  ({isaretli})")
print("=" * 38)

# Grafik
fig = build_performance_line_chart(
    portfolio_series=None,
    benchmark_series=benchmark_df,
    currency_label="TL (Nominal)",
    title=f"Son 30 Is Gunu Benchmark ({actual_start} - {end_date})",
)
display(fig)

---
## CSV Veri Doğrulama Testleri

Aşağıdaki hücreler `data/` klasöründeki CSV dosyalarının yapısını ve içeriğini doğrular.
Her test PASS veya açıklayıcı hata mesajı verir.

In [ ]:
# portfolio.csv - yapi ve icerik dogrulama
import pandas as pd, os

DATA_DIR = os.path.join(os.getcwd(), 'data')
path = os.path.join(DATA_DIR, 'portfolio.csv')

if not os.path.exists(path):
    print(f'portfolio.csv YOK ({path}) - yeni kullanici icin normal.')
    print('PortfolyoBenchmark.ipynb Cell-6 formundan ilk islem ekleyince olusur.')
else:
    df = pd.read_csv(path)
    hatalar = []

    # Sutun varligi
    def _norm(c):
        return (c.replace('ı','i').replace('ş','s').replace('Ş','S')
                 .replace('ü','u').replace('ö','o').replace('ç','c')
                 .replace('ğ','g').replace('İ','I').replace('Ü','U'))
    normalized = [_norm(c) for c in df.columns]
    for col in ['Varlik Adi', 'Alis Tarihi', 'Alis Fiyati', 'Miktar', 'Komisyon']:
        if col not in normalized:
            hatalar.append(f'Eksik sutun: {col}')

    if len(df) == 0:
        hatalar.append('portfolio.csv bos')

    # Sayisal sutunlar
    for col_idx, col_name in enumerate(df.columns):
        if _norm(col_name) in ['Alis Fiyati', 'Miktar', 'Komisyon']:
            try:
                vals = pd.to_numeric(df.iloc[:, col_idx], errors='raise')
                if (vals < 0).any():
                    hatalar.append(f'{col_name} sutununda negatif deger var')
            except Exception:
                hatalar.append(f'{col_name} sutunu sayisal degil')

    # Tarih parse
    try:
        tarih_col = [c for c in df.columns if 'arih' in c][0]
        parsed = pd.to_datetime(df[tarih_col], dayfirst=True, errors='coerce')
        null_count = parsed.isna().sum()
        if null_count > 0:
            hatalar.append(f'{null_count} tarih parse edilemedi (GG.AA.YYYY format?)')
        elif (parsed > pd.Timestamp.today()).any():
            hatalar.append('Gelecek tarihli alis var - kontrol et')
    except Exception as e:
        hatalar.append(f'Tarih sutunu bulunamadi: {e}')

    print(f'portfolio.csv - {len(df)} pozisyon, {len(df.columns)} sutun')
    if hatalar:
        for h in hatalar:
            print(f'  HATA: {h}')
    else:
        print('  PASS: Yapi ve icerik gecerli')
    print(df.to_string(index=False))

In [ ]:
# transactions.csv - yapi, gecerli islem turleri, siralama, negatif deger
path = os.path.join(DATA_DIR, 'transactions.csv')

if not os.path.exists(path):
    print(f'transactions.csv YOK ({path}) - yeni kullanici icin normal.')
    print('PortfolyoBenchmark.ipynb Cell-6 formundan ilk islem ekleyince olusur.')
else:
    df = pd.read_csv(path)
    VALID_ISLEM = {'ALIS', 'SATIS', 'NAKIT_GIRIS', 'NAKIT_CIKIS'}
    hatalar = []

    if len(df) == 0:
        hatalar.append('transactions.csv bos')

    col_clean = lambda c: (c.replace('ı','i').replace('ş','s').replace('Ş','S')
                            .replace('ü','u').replace('ö','o').replace('ç','c')
                            .replace('ğ','g').replace('İ','I').replace('Ü','U'))
    clean_cols = [col_clean(c) for c in df.columns]
    for req in ['Tarih', 'Varlik Adi', 'Islem Turu', 'Fiyat', 'Miktar', 'Komisyon']:
        if req not in clean_cols:
            hatalar.append(f'Eksik sutun: {req}')

    # Islem turleri
    islem_col = df.columns[[col_clean(c) == 'Islem Turu' for c in df.columns]]
    if len(islem_col) > 0:
        gecersiz = set(df[islem_col[0]].astype(str).map(col_clean)) - VALID_ISLEM
        if gecersiz:
            hatalar.append(f'Gecersiz Islem Turu degerleri: {gecersiz}')

    # Sayisal kontroller
    for col_idx, c in enumerate(df.columns):
        if col_clean(c) in ['Fiyat', 'Miktar', 'Komisyon']:
            try:
                vals = pd.to_numeric(df.iloc[:, col_idx], errors='raise')
                if (vals < 0).any():
                    hatalar.append(f'{c} sutununda negatif deger')
            except Exception:
                hatalar.append(f'{c} sayisal degil')

    # Tarih parse ve siralama
    try:
        tarih_col = [c for c in df.columns if 'arih' in c][0]
        parsed = pd.to_datetime(df[tarih_col], dayfirst=True, errors='coerce')
        null_count = parsed.isna().sum()
        if null_count > 0:
            hatalar.append(f'{null_count} tarih parse edilemedi')
        # load_transactions_csv otomatik sirali - siralama check info amacli
        if (parsed > pd.Timestamp.today()).any():
            hatalar.append('Gelecek tarihli islem var')
    except Exception as e:
        hatalar.append(f'Tarih parse hatasi: {e}')

    print(f'transactions.csv - {len(df)} islem, {len(df.columns)} sutun')
    if hatalar:
        for h in hatalar:
            print(f'  HATA: {h}')
    else:
        print('  PASS: Yapi, islem turleri ve sayisal degerler gecerli')
    print(df.to_string(index=False))

In [ ]:
# cpi_turkey.csv - yapi, tarih araligi, YILLIK YoY artis, bosluk kontrolu
# NOT: TUFE mevsimsel (yaz aylarinda gida fiyatlari) sebebiyle aylik bazda
# dusebilir. Bu normal Turkiye verisi - is_monotonic_increasing yanlis pozitif
# verir. Bunun yerine yillik YoY artisi kontrol edilir.
path = os.path.join(DATA_DIR, 'cpi_turkey.csv')
df = pd.read_csv(path)
hatalar = []

if 'Tarih' not in df.columns:
    hatalar.append('Eksik sutun: Tarih')
if 'CPI_Endeks' not in df.columns:
    hatalar.append('Eksik sutun: CPI_Endeks')

if not hatalar:
    df['Tarih'] = pd.to_datetime(df['Tarih'], dayfirst=True, errors='coerce')
    null_count = df['Tarih'].isna().sum()
    if null_count > 0:
        hatalar.append(f'{null_count} tarih parse edilemedi')

    vals = pd.to_numeric(df['CPI_Endeks'], errors='coerce')
    if vals.isna().any():
        hatalar.append('CPI_Endeks sutununda sayisal olmayan deger')
    if (vals <= 0).any():
        hatalar.append('CPI_Endeks sifir veya negatif deger iceriyor')

    df_sorted = df.dropna(subset=['Tarih']).sort_values('Tarih').reset_index(drop=True)

    # YILLIK YoY check: her takvim yili icin son endeks, oncekinden buyuk olmali
    yearly_last = df_sorted.set_index('Tarih')['CPI_Endeks'].groupby(
        df_sorted['Tarih'].dt.year.values
    ).last()
    yoy = yearly_last.pct_change().dropna()
    if (yoy < 0).any():
        bad_years = yoy[yoy < 0].index.tolist()
        hatalar.append(
            f'CPI yillik bazda gerileyen yil(lar): {bad_years} - TUFE yillik negatif olmamali'
        )

    # Bosluk: iki satir arasi 60 gunden fazla olmamali
    bosluklar = df_sorted['Tarih'].diff().dt.days.dropna()
    if len(bosluklar) > 0:
        max_bosluk = bosluklar.max()
        if max_bosluk > 60:
            hatalar.append(
                f'Veri boslugu {max_bosluk:.0f} gun - 60 gun siniri asildi (kayip ay olabilir)'
            )

    # Guncellik
    son_tarih = df_sorted['Tarih'].max()
    gecikme = (pd.Timestamp.today() - son_tarih).days
    if gecikme > 90:
        hatalar.append(f'CPI verisi {gecikme} gundur guncellenmemis (son: {son_tarih.date()})')

print(f'cpi_turkey.csv - {len(df)} aylik kayit')
print(f'  Aralik : {df["Tarih"].min().date()} - {df["Tarih"].max().date()}')
print(f'  CPI    : {vals.min():.2f} - {vals.max():.2f}')
if hatalar:
    for h in hatalar:
        print(f'  HATA: {h}')
else:
    print('  PASS: Yapi, yillik YoY ve guncellik gecerli')

In [ ]:
# tcmb_rates.csv — yapı, geçerli oran aralığı, kronoloji
path = os.path.join(DATA_DIR, 'tcmb_rates.csv')
hatalar = []

if not os.path.exists(path):
    print('tcmb_rates.csv YOK — sabit oran (Tier 3 fallback) kullanilacak. Bu normal.')
else:
    df = pd.read_csv(path)

    # Sütunlar
    if 'Tarih' not in df.columns:
        hatalar.append('Eksik sutun: Tarih')
    if 'Faiz_Orani_Yillik_Pct' not in df.columns:
        hatalar.append('Eksik sutun: Faiz_Orani_Yillik_Pct')

    if not hatalar:
        df['Tarih'] = pd.to_datetime(df['Tarih'], dayfirst=True, errors='coerce')
        null_count = df['Tarih'].isna().sum()
        if null_count > 0:
            hatalar.append(f'{null_count} tarih parse edilemedi')

        rates = pd.to_numeric(df['Faiz_Orani_Yillik_Pct'], errors='coerce')
        if rates.isna().any():
            hatalar.append('Faiz_Orani_Yillik_Pct sayisal olmayan deger iceriyor')
        if (rates < 0).any():
            hatalar.append('Negatif faiz orani var')
        if (rates > 100).any():
            hatalar.append('Faiz orani >100 — yuzde formatinda olmali')
        if (rates == 0).any():
            hatalar.append('Sifir faiz orani var — yanlis giris olabilir')

        # Kronoloji
        df_sorted = df.dropna(subset=['Tarih']).sort_values('Tarih')
        if not df_sorted['Tarih'].is_monotonic_increasing:
            hatalar.append('Satirlar kronolojik degil')

        # Güncellik: son karardan bu yana 180 günden fazla geçtiyse uyar
        son_tarih = df_sorted['Tarih'].max()
        gecikme = (pd.Timestamp.today() - son_tarih).days
        if gecikme > 180:
            hatalar.append(f'Faiz verisi {gecikme} gundur guncellenmemis (son: {son_tarih.date()})')

        print(f'tcmb_rates.csv — {len(df)} karar degisikligi')
        print(f'  Aralik : {df_sorted["Tarih"].min().date()} — {son_tarih.date()}')
        print(f'  Oran   : {rates.min():.1f}% — {rates.max():.1f}%')
        if hatalar:
            for h in hatalar:
                print(f'  HATA: {h}')
        else:
            print('  PASS: Yapi ve oran araligi gecerli')
    print(df.to_string(index=False))

In [ ]:
# deposit_rates.csv + mevduat benchmark dogrulama
import os, sys, pandas as pd
from datetime import datetime, timedelta

DATA_DIR = os.path.join(os.getcwd(), "data")
LIB_PATH = os.path.join(os.getcwd(), "lib")
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from data_loader import load_deposit_rates, load_tcmb_rates
from benchmark_engine import build_deposit_series

TCMB_PATH    = os.path.join(DATA_DIR, "tcmb_rates.csv")
DEPOSIT_PATH = os.path.join(DATA_DIR, "deposit_rates.csv")
hatalar = []

# 1. CSV yapi kontrolu
if not os.path.exists(DEPOSIT_PATH):
    print("deposit_rates.csv YOK - load_deposit_rates() ile ilk scrape henuz yapilmamis.")
    print("Notebook Cell-5'te load_deposit_rates() calistiginda otomatik olusur.")
else:
    df = pd.read_csv(DEPOSIT_PATH)

    for col in ["Tarih", "Faiz_Orani_Yillik_Pct"]:
        if col not in df.columns:
            hatalar.append(f"Eksik sutun: {col}")

    if not hatalar:
        df["Tarih"] = pd.to_datetime(df["Tarih"], dayfirst=True, errors="coerce")
        null_tarih = df["Tarih"].isna().sum()
        if null_tarih > 0:
            hatalar.append(f"{null_tarih} tarih parse edilemedi (GG.AA.YYYY bekleniyor)")

        rates = pd.to_numeric(df["Faiz_Orani_Yillik_Pct"], errors="coerce")
        if rates.isna().any():
            hatalar.append("Faiz_Orani_Yillik_Pct sayisal olmayan deger iceriyor")
        if (rates < 0).any() or (rates > 200).any():
            hatalar.append(f"Faiz orani makul aralik disinda: min={rates.min():.1f}  max={rates.max():.1f}")

        df_s = df.dropna(subset=["Tarih"]).sort_values("Tarih")
        son_tarih = df_s["Tarih"].max()
        gecikme = (pd.Timestamp.today() - son_tarih).days
        if gecikme > 90:
            hatalar.append(f"Veri {gecikme} gundur guncellenmemis (son: {son_tarih.date()})")

        print(f"deposit_rates.csv - {len(df)} kayit")
        print(f"  Aralik  : {df_s['Tarih'].min().date()} - {son_tarih.date()}")
        print(f"  Oran    : {rates.min():.2f}% - {rates.max():.2f}%")
        print(f"  Son     : {rates.iloc[-1]:.2f}%  ({son_tarih.date()})")
        print()

    # 2. load_deposit_rates() servis testi
    tcmb_rates    = load_tcmb_rates(TCMB_PATH, policy_rate_pct=37, auto_refresh=False)
    deposit_rates = load_deposit_rates(DEPOSIT_PATH, fallback_policy_rate_series=tcmb_rates, auto_refresh=False)

    print("load_deposit_rates() gunluk seri:")
    print(f"  Uzunluk  : {len(deposit_rates)} gun")
    print(f"  Aralik   : {deposit_rates.index[0].date()} - {deposit_rates.index[-1].date()}")
    print(f"  Son oran : {deposit_rates.iloc[-1]:.2f}%")
    print()

    # 3. Politika faizi vs mevduat kiyaslamasi
    p_son = tcmb_rates.iloc[-1]
    m_son = deposit_rates.iloc[-1]
    fark  = m_son - p_son
    print(f"  Politika faizi (tcmb_rates)   son: {p_son:.2f}%")
    print(f"  Mevduat  faizi (deposit_rates) son: {m_son:.2f}%")
    print(f"  Fark: {'+' if fark >= 0 else ''}{fark:.2f} puan  ({'mevduat daha yuksek' if fark > 0 else 'politika daha yuksek'})")
    print()

    # 4. build_deposit_series() endeks sanity check
    end_dt   = datetime.today().strftime("%Y-%m-%d")
    start_dt = (datetime.today() - timedelta(days=365)).strftime("%Y-%m-%d")

    endeks_p = build_deposit_series(tcmb_rates,    start_dt, end_dt)
    endeks_m = build_deposit_series(deposit_rates, start_dt, end_dt)

    assert abs(endeks_p.iloc[0] - 100.0) < 1e-6, f"Politika endeksi baz=100 baslamali: {endeks_p.iloc[0]}"
    assert abs(endeks_m.iloc[0] - 100.0) < 1e-6, f"Mevduat endeksi baz=100 baslamali: {endeks_m.iloc[0]}"
    assert endeks_p.iloc[-1] > 100, "1 yil boyunca pozitif getiri bekleniyor (politika)"
    assert endeks_m.iloc[-1] > 100, "1 yil boyunca pozitif getiri bekleniyor (mevduat)"

    print(f"build_deposit_series() 1 yillik endeks ({start_dt} - {end_dt}):")
    print(f"  Politika faizi : baslangic=100  son={endeks_p.iloc[-1]:.2f}  (+{endeks_p.iloc[-1]-100:.2f})")
    print(f"  Mevduat  faizi : baslangic=100  son={endeks_m.iloc[-1]:.2f}  (+{endeks_m.iloc[-1]-100:.2f})")
    print()

    if hatalar:
        for h in hatalar:
            print(f"  HATA: {h}")
    else:
        print("PASS: deposit_rates.csv yapi + load_deposit_rates + build_deposit_series testleri gecti")

In [ ]:
# Capraz tutarlilik - portfolio (yatirilan + benchmark) vs transactions (gercek alislar)
# Benchmark-only varliklar (BIST100, DOLAR, EURO, Gram Altin/Gumus, Mevduat)
# karsilastirma amaclidir; transactions ALIS kaydi olmayabilir. Test bunlari haric tutar.
BENCHMARK_ONLY = {"BIST100", "DOLAR", "EURO", "Gram Altin", "Gram Gumus", "Mevduat"}

port_path  = os.path.join(DATA_DIR, 'portfolio.csv')
trans_path = os.path.join(DATA_DIR, 'transactions.csv')

if not (os.path.exists(port_path) and os.path.exists(trans_path)):
    print(f'portfolio.csv veya transactions.csv YOK - yeni kullanici icin normal. Capraz kontrol atlandi.')
else:
    hatalar = []
    port  = pd.read_csv(port_path)
    trans = pd.read_csv(trans_path)

    # Sutun adlari encode bagimsiz al
    varlik_col_p = [c for c in port.columns  if 'arl' in c][0]
    varlik_col_t = [c for c in trans.columns if 'arl' in c][0]
    islem_col_t  = [c for c in trans.columns if 'lem' in c or 'sl' in c][0]

    port_varliklar  = set(port[varlik_col_p].dropna().astype(str).unique())
    # ALIS satirlarindaki varlik adlari (encode-tolerant)
    alis_mask = trans[islem_col_t].astype(str).str.contains('ALI', na=False)
    trans_varliklar = set(trans.loc[alis_mask, varlik_col_t].dropna().astype(str).unique())

    # Portfolyoda olup ALIS'ta olmayan + benchmark-only haric
    kayip = (port_varliklar - trans_varliklar) - BENCHMARK_ONLY
    if kayip:
        hatalar.append(f'Portfolyodaki gercek varliklar transactions ALIS kaydinda yok: {kayip}')

    # Fiyat/Miktar sayisal
    miktar_col = [c for c in trans.columns if 'iktar' in c][0]
    fiyat_col  = [c for c in trans.columns if 'iyat' in c][0]
    if pd.to_numeric(trans[fiyat_col], errors='coerce').isna().any():
        hatalar.append('transactions.csv Fiyat sutununda sayisal olmayan deger')
    if pd.to_numeric(trans[miktar_col], errors='coerce').isna().any():
        hatalar.append('transactions.csv Miktar sutununda sayisal olmayan deger')

    print('Capraz tutarlilik kontrolu:')
    print(f'  Portfolio varliklar       : {sorted(port_varliklar)}')
    print(f'  Transactions ALIS         : {sorted(trans_varliklar)}')
    print(f'  Benchmark-only (haric)    : {sorted(BENCHMARK_ONLY & port_varliklar)}')
    if hatalar:
        for h in hatalar:
            print(f'  HATA: {h}')
    else:
        print('  PASS: Varlik adlari tutarli (benchmark-only haric tutuldu)')

print()
print('=' * 40)
print('CSV TEST OZETI')
print('  Yukarida PASS gorunuyorsa tum CSV dosyalari gecerli.')
print('  HATA gorunuyorsa ilgili dosyayi guncelle.')
print('=' * 40)

In [ ]:
# WAC commission simetri testi - compute_wac() ALIS ve SATIS komisyonlarini
# birim basina dagitarak simetrik calismali. Sentetik tx ile dogrula.
import pandas as pd
import sys, os
LIB_PATH = os.path.join(os.getcwd(), "lib")
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)
from portfolio_engine import compute_wac

# Senaryo: 100 birim X @ 100 TL (komisyon 20 TL toplam) -> WAC = 100 + 20/100 = 100.2
# Sonra 50 birim satis @ 110 TL (komisyon 2 TL toplam)
#   Net satis fiyati/birim = 110 - 2/50 = 109.96
#   Realized PnL = (109.96 - 100.2) * 50 = 488.0
tx = pd.DataFrame([
    {"Tarih": pd.Timestamp("2025-01-01"), "Varlık Adı": "X", "İşlem Türü": "ALIŞ",
     "Fiyat": 100.0, "Miktar": 100.0, "Komisyon": 20.0},
    {"Tarih": pd.Timestamp("2025-02-01"), "Varlık Adı": "X", "İşlem Türü": "SATIŞ",
     "Fiyat": 110.0, "Miktar": 50.0,  "Komisyon": 2.0},
])
state = compute_wac(tx)
assert abs(state["X"]["wac"] - 100.2) < 1e-9, f'WAC bekleniyor 100.2, alindi: {state["X"]["wac"]}'
assert abs(state["X"]["realized_pnl"] - 488.0) < 1e-9, f'realized_pnl bekleniyor 488.0, alindi: {state["X"]["realized_pnl"]}'
assert state["X"]["units"] == 50.0, f'units bekleniyor 50, alindi: {state["X"]["units"]}'

# Komisyon=0 edge case: realized = (110 - 100) * 50 = 500
tx2 = pd.DataFrame([
    {"Tarih": pd.Timestamp("2025-01-01"), "Varlık Adı": "Y", "İşlem Türü": "ALIŞ",
     "Fiyat": 100.0, "Miktar": 100.0, "Komisyon": 0.0},
    {"Tarih": pd.Timestamp("2025-02-01"), "Varlık Adı": "Y", "İşlem Türü": "SATIŞ",
     "Fiyat": 110.0, "Miktar": 50.0,  "Komisyon": 0.0},
])
state2 = compute_wac(tx2)
assert abs(state2["Y"]["wac"] - 100.0) < 1e-9
assert abs(state2["Y"]["realized_pnl"] - 500.0) < 1e-9

print("PASS: WAC commission simetri (ALIS ve SATIS komisyonu birim basina dagitilmali)")
print(f"  Test 1 (komisyonlu): WAC={state['X']['wac']}, realized_pnl={state['X']['realized_pnl']}")
print(f"  Test 2 (komisyonsuz): WAC={state2['Y']['wac']}, realized_pnl={state2['Y']['realized_pnl']}")